## Notebook 09: Semantic Matching Hyperparameter Choice

This notebook analyses the labeled data from notebook 08 to select the best embedding model and the optimal similarity delta threshold for the SENSE matching step.

**What it does:**
1. Reads the labeled audit workbook from notebook 08.
2. Computes recall at K, mean reciprocal rank, and mean plausible rank for each model on dev and test splits.
3. Sweeps over similarity delta values for the chosen model (BGE) and evaluates precision, recall, and a regularised utility metric.
4. Selects the optimal delta that maximises utility on the dev set and reports test set performance.
5. Produces publication-ready figures comparing models and delta choices.

**Inputs:**
- `semantic_matching_audit_labeled.xlsx` from notebook 08.
- PSILCA sector list JSON.

**Outputs:**
- Figures (PDF and PNG) in the output directory: model comparison, delta sweep, and selected operating point.
- Console output with optimal delta value and performance metrics.

In [ ]:
# %% [markdown]
# # Semantic matching audit — Notebook B (analysis) — IJLCA-ready plots (UPDATED: Δ via regularized utility)
#
# Layout goal (all figures):
#   (a) Subtitle   (b) Subtitle   (c) Subtitle   (CENTERED above each panel)
#     [small space]
#   plots...
#     [small space]
#   legend (below)
#
# Styling rule you requested:
#   - DEV = solid throughout
#   - TEST = hatched throughout
#   - Model colors + model order are consistent across all plots

# %%
from __future__ import annotations

import os
import json
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MaxNLocator
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

from sentence_transformers import SentenceTransformer


# %% [markdown]
# ## 0) Inputs

# %%
OUT_DIR = r"OUT_DIR"
PSILCA_SECTORS_JSON = r"PSILCA_SECTORS_JSON"
LABELED_XLSX = os.path.join(OUT_DIR, "semantic_matching_audit_labeled.xlsx")

TFIDF_BASELINE = "baseline/tfidf_char_wb_3_5"

MODELS_SEMANTIC = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "BAAI/bge-base-en-v1.5",
    "sentence-transformers/all-mpnet-base-v2",
]
MODELS_ALL = MODELS_SEMANTIC + [TFIDF_BASELINE]

# Must match your workbook labeling range (top-10)
TOP_K = 10
LOW_RES_MAX = 26

# Δ sweep parameters (chosen model only)
DELTA_MIN  = 0.005
DELTA_MAX  = 0.050
DELTA_STEP = 0.0001
DELTA_GRID = np.round(np.arange(DELTA_MIN, DELTA_MAX + 1e-12, DELTA_STEP), 6)

# Embedding settings for Δ sweep
DEVICE = None
BATCH_SIZE = 256

# BGE prompting (must match what you used in matching)
BGE_MODEL = "BAAI/bge-base-en-v1.5"
BGE_MODE = "intended"
BGE_QUERY_PROMPT = "Represent this sentence for searching relevant passages: "

# Δ selection regularization strength (METHOD 1)
LAMBDA_EXTRA = 0.10

# Save figures? (off by default)
SAVE_FIGS = True
FIG_DPI = 1200


def safe_model_key(model_name: str) -> str:
    return model_name.replace("/", "__").replace(":", "_")


def prep_plain(text: str) -> str:
    return (text or "").strip()


def prep_bge_query_intended(text: str) -> str:
    return BGE_QUERY_PROMPT + (text or "").strip()


def get_preprocessors(model_name: str):
    if model_name == BGE_MODEL and BGE_MODE == "intended":
        return prep_bge_query_intended, prep_plain
    return prep_plain, prep_plain


LABELS = {
    "sentence-transformers/all-MiniLM-L6-v2": "MiniLM-L6",
    "BAAI/bge-base-en-v1.5": "BGE-base",
    "sentence-transformers/all-mpnet-base-v2": "MPNet-base",
    TFIDF_BASELINE: "TF-IDF char(3–5)",
}

# Canonical model order (consistent everywhere)
MODEL_ORDER_ALL = [TFIDF_BASELINE] + MODELS_SEMANTIC
MODEL_ORDER_SEM = list(MODELS_SEMANTIC)


# %% [markdown]
# ## 1) Load PSILCA sectors + labeled workbook

# %%
def read_json(path: str) -> Any:
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


psilca_sectors: Dict[str, List[str]] = read_json(PSILCA_SECTORS_JSON)
psilca_sectors = {k.strip().upper(): v for k, v in psilca_sectors.items()}

if not os.path.exists(LABELED_XLSX):
    raise FileNotFoundError(f"Missing labeled workbook: {LABELED_XLSX}")

xls = pd.ExcelFile(LABELED_XLSX)
print("Sheets:", len(xls.sheet_names))


# %% [markdown]
# ## 2) Parse labels and compute per-activity metric contributions

# %%
def parse_plausible_ranks(s: Any, k: int) -> List[int]:
    """Parse '1,3,7' -> sorted unique integer ranks within [1..k]."""
    if s is None:
        return []
    s = str(s).strip()
    if not s:
        return []
    out = []
    for part in s.split(","):
        part = part.strip()
        if part.isdigit():
            r = int(part)
            if 1 <= r <= k:
                out.append(r)
    return sorted(set(out))


def best_rank(ranks: List[int]) -> Optional[int]:
    return min(ranks) if ranks else None


def load_labels_for_models(labeled_xlsx: str, models: List[str]) -> pd.DataFrame:
    rows = []
    for m in models:
        sheet = f"{safe_model_key(m)}_top10"
        if sheet not in xls.sheet_names:
            raise KeyError(f"Missing sheet '{sheet}' in labeled workbook.")
        df = pd.read_excel(labeled_xlsx, sheet_name=sheet)

        df["plausible_rank_list"] = df["plausible_ranks"].apply(lambda x: parse_plausible_ranks(x, TOP_K))
        df["best_rank"] = df["plausible_rank_list"].apply(best_rank)

        df["success"] = df["best_rank"].notna().astype(int)
        df["top1"] = (df["best_rank"] == 1).astype(int)
        df["mrr_contrib"] = df["best_rank"].apply(
            lambda r: 1.0 / r if (r is not None and np.isfinite(r)) else 0.0
        )

        df["stratum"] = np.where(df["n_sectors"] <= LOW_RES_MAX, "low_res", "high_res")

        rows.append(df[[
            "model","activity_id","split","psilca_country","n_sectors","stratum",
            "best_rank","success","top1","mrr_contrib"
        ]])
    return pd.concat(rows, ignore_index=True)


labels = load_labels_for_models(LABELED_XLSX, MODELS_ALL)
print("Loaded labels:", labels.shape)

n_dev_acts  = labels[labels["split"] == "dev"]["activity_id"].nunique()
n_test_acts = labels[labels["split"] == "test"]["activity_id"].nunique()
print(f"Unique activities: DEV n={n_dev_acts}, TEST n={n_test_acts}")


# %% [markdown]
# ## 3) Summaries (mean over activities; no CIs)

# %%
def summarize(df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    out = []
    for keys, sub in df.groupby(group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {k: v for k, v in zip(group_cols, keys)}
        row["n"] = int(len(sub))
        row["top1"] = float(sub["top1"].mean())
        row[f"success@{TOP_K}"] = float(sub["success"].mean())
        row["MRR"] = float(sub["mrr_contrib"].mean())
        out.append(row)
    return pd.DataFrame(out)


overall = summarize(labels, ["model", "split"])


# %% [markdown]
# ## 4) Plot helpers (IJLCA-ish + your palette + consistent panel headers + legend below)

# %%
plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.9,
    "grid.linewidth": 0.6,
    "lines.linewidth": 1.8,
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# --- Colors (from your other script) ---
BASE_PALETTE = {
    "MiniLM-L6": "#0072B2",
    "BGE-base":  "#009E73",
    "MPNet-base":"#E69F00",
    "TF-IDF char(3–5)": "#666666",
}

def extend_palette_for_used_labels(used_labels: List[str], base_palette: Dict[str, str]) -> Dict[str, str]:
    pal = dict(base_palette)
    fallback_colors = ["#666666", "#8c8c8c", "#4d4d4d", "#b3b3b3"]
    i = 0
    for lbl in sorted(set(used_labels)):
        if lbl not in pal and pd.notna(lbl):
            pal[lbl] = fallback_colors[i % len(fallback_colors)]
            i += 1
    return pal

# Fig 3 split colors (your palette)
SPLIT_COLORS = {
    "DEV":  "#0072B2",
    "TEST": "#D55E00",
}

# "DEV solid, TEST hatched" convention
HATCH_TEST = "//"

# Layout controls
FIG_TOP = 0.84
FIG_BOTTOM = 0.38
LEGEND_Y = 0.07
PANEL_TITLE_PAD = 10

def nice_axes(ax):
    ax.set_axisbelow(True)
    ax.grid(True, which="major", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))

def rate_axis(ax):
    ax.set_ylim(0, 1.0)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))

def save_figure(fig, basename):
    if not SAVE_FIGS:
        return
    pdf_path = os.path.join(OUT_DIR, f"{basename}.pdf")
    png_path = os.path.join(OUT_DIR, f"{basename}.png")
    fig.savefig(pdf_path, bbox_inches="tight")
    fig.savefig(png_path, dpi=FIG_DPI, bbox_inches="tight")
    print("Saved:", pdf_path)

def panel_header(ax, letter: str, subtitle: str, pad: int = PANEL_TITLE_PAD):
    """Centered header: '(a) Subtitle' (same font as axes titles)."""
    ax.set_title(f"({letter}) {subtitle}", loc="center", pad=pad)

def add_legend_below(fig, handles, labels, ncol: int, title: Optional[str] = None, y: float = LEGEND_Y):
    leg = fig.legend(
        handles, labels,
        loc="lower center",
        bbox_to_anchor=(0.5, y),
        ncol=ncol,
        frameon=False,
        title=title,
        title_fontsize=plt.rcParams.get("legend.fontsize", 9),
    )
    return leg


# %% [markdown]
# ## 5) FIGURE 1 (main text): Why semantic matching? (TEST only; baseline vs semantic)

# %%
def plot_fig1_why_semantic(overall_df: pd.DataFrame):
    metrics = [
        ("top1", "Top-1 plausibility", "Rate", "a"),
        (f"success@{TOP_K}", f"Plausible-hit@{TOP_K}", "Rate", "b"),
        ("MRR", "MRR (plausible)", "MRR", "c"),
    ]

    dfp = overall_df[(overall_df["split"] == "test") & (overall_df["model"].isin(MODEL_ORDER_ALL))].copy()
    dfp = dfp.set_index("model").loc[MODEL_ORDER_ALL].reset_index()

    xlabels = [LABELS.get(m, m) for m in dfp["model"]]
    palette = extend_palette_for_used_labels(xlabels, BASE_PALETTE)

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), constrained_layout=False)
    fig.subplots_adjust(top=FIG_TOP, bottom=FIG_BOTTOM, wspace=0.35)

    for ax, (col, subtitle, ylab, letter) in zip(axes, metrics):
        x = np.arange(len(xlabels))
        y = dfp[col].to_numpy(dtype=float)

        bars = ax.bar(x, y, edgecolor="black", linewidth=0.8)

        # TEST-only figure => hatch everything (TEST convention)
        for i, b in enumerate(bars):
            b.set_facecolor(palette.get(xlabels[i], "#999999"))
            b.set_hatch(HATCH_TEST)

        ax.set_xticks(x)
        ax.set_xticklabels(xlabels, rotation=15, ha="right")

        panel_header(ax, letter, subtitle)
        ax.set_ylabel(ylab)

        if col in ("top1", f"success@{TOP_K}"):
            rate_axis(ax)
        else:
            ax.set_ylim(0, 1.0)

        nice_axes(ax)

    # Legend below: model colors + TEST hatch (consistent)
    handles = [
        Patch(facecolor=palette.get(lbl, "#999999"), edgecolor="black", linewidth=0.8, hatch=HATCH_TEST, label=lbl)
        for lbl in xlabels
    ]
    add_legend_below(
        fig,
        handles=handles,
        labels=[h.get_label() for h in handles],
        ncol=min(4, len(handles)),
        title=f"TEST (n={n_test_acts})",
        y=LEGEND_Y
    )

    save_figure(fig, "Fig1_semantic_vs_baseline_TEST")
    plt.show()


plot_fig1_why_semantic(overall)


# %% [markdown]
# ## 6) FIGURE 2 (SI): Semantic model choice (DEV vs TEST; semantic models only)

# %%
def plot_fig2_model_choice(overall_df: pd.DataFrame):
    metrics = [
        ("top1", "Top-1 plausibility", "Rate", "a"),
        (f"success@{TOP_K}", f"Plausible-hit@{TOP_K}", "Rate", "b"),
        ("MRR", "MRR (plausible)", "MRR", "c"),
    ]

    df_sem = overall_df[overall_df["model"].isin(MODEL_ORDER_SEM)].copy()
    # enforce consistent order (not ranked)
    df_sem["model"] = pd.Categorical(df_sem["model"], categories=MODEL_ORDER_SEM, ordered=True)

    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), constrained_layout=False)
    fig.subplots_adjust(top=FIG_TOP, bottom=FIG_BOTTOM, wspace=0.35)

    xlabels = [LABELS.get(m, m) for m in MODEL_ORDER_SEM]
    palette = extend_palette_for_used_labels(xlabels, BASE_PALETTE)
    x = np.arange(len(xlabels))
    width = 0.36

    for ax, (col, subtitle, ylab, letter) in zip(axes, metrics):
        p = df_sem.pivot(index="model", columns="split", values=col).loc[MODEL_ORDER_SEM]

        y_dev = p["dev"].to_numpy(dtype=float)
        y_tst = p["test"].to_numpy(dtype=float)

        # DEV solid, TEST hatched (throughout)
        b_dev = ax.bar(x - width/2, y_dev, width, edgecolor="black", linewidth=0.8)
        b_tst = ax.bar(x + width/2, y_tst, width, edgecolor="black", linewidth=0.8)

        for i in range(len(xlabels)):
            color = palette.get(xlabels[i], "#999999")
            b_dev[i].set_facecolor(color)
            b_tst[i].set_facecolor(color)
            b_tst[i].set_hatch(HATCH_TEST)

        ax.set_xticks(x)
        ax.set_xticklabels(xlabels, rotation=15, ha="right")

        panel_header(ax, letter, subtitle)
        ax.set_ylabel(ylab)

        if col in ("top1", f"success@{TOP_K}"):
            rate_axis(ax)
        else:
            ax.set_ylim(0, 1.0)

        nice_axes(ax)

    # Legend below: encode split styling (DEV solid vs TEST hatched)
    split_handles = [
        Patch(facecolor="white", edgecolor="black", hatch="", label="DEV (solid)"),
        Patch(facecolor="white", edgecolor="black", hatch=HATCH_TEST, label="TEST (hatched)"),
    ]
    add_legend_below(
        fig,
        handles=split_handles,
        labels=[h.get_label() for h in split_handles],
        ncol=2,
        title=f"DEV n={n_dev_acts}, TEST n={n_test_acts}",
        y=LEGEND_Y
    )

    save_figure(fig, "Fig2_semantic_model_choice_DEV_TEST")
    plt.show()


plot_fig2_model_choice(overall)


# %% [markdown]
# ## 7) Select CHOSEN_MODEL on DEV among semantic models (MRR → Top-1 → Success@K)

# %%
dev_sem = overall[(overall["split"] == "dev") & (overall["model"].isin(MODELS_SEMANTIC))].copy()
dev_sem = dev_sem.sort_values(["MRR", "top1", f"success@{TOP_K}"], ascending=[False, False, False])
CHOSEN_MODEL = dev_sem.iloc[0]["model"]
print("Chosen semantic model:", CHOSEN_MODEL)


# %% [markdown]
# ## 8) Δ sweep (chosen model, high_res) + regularized utility selection on DEV

# %%
sheet = f"{safe_model_key(CHOSEN_MODEL)}_top10"
dfM = pd.read_excel(LABELED_XLSX, sheet_name=sheet)

dfM["plausible_rank_list"] = dfM["plausible_ranks"].apply(lambda x: parse_plausible_ranks(x, TOP_K))
dfM["has_plausible_in_topk"] = dfM["plausible_rank_list"].apply(len) > 0
dfM["stratum"] = np.where(dfM["n_sectors"] <= LOW_RES_MAX, "low_res", "high_res")

dev_high  = dfM[(dfM["split"] == "dev")  & (dfM["stratum"] == "high_res")].copy()
test_high = dfM[(dfM["split"] == "test") & (dfM["stratum"] == "high_res")].copy()

print("DEV high_res:", len(dev_high), "with plausible-in-topK:", int(dev_high["has_plausible_in_topk"].sum()))
print("TEST high_res:", len(test_high), "with plausible-in-topK:", int(test_high["has_plausible_in_topk"].sum()))

model = SentenceTransformer(CHOSEN_MODEL, device=DEVICE)
prep_act, prep_sec = get_preprocessors(CHOSEN_MODEL)

def embed_texts(texts: List[str], preprocess) -> np.ndarray:
    texts_clean = [preprocess(t) for t in texts]
    emb = model.encode(
        texts_clean,
        batch_size=BATCH_SIZE,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    return emb.astype(np.float32)

countries = sorted(set(dev_high["psilca_country"].astype(str)) | set(test_high["psilca_country"].astype(str)))
sector_emb_by_country: Dict[str, Tuple[List[str], np.ndarray]] = {}
for c in countries:
    c2 = c.strip().upper()
    names = list(psilca_sectors[c2])
    emb = embed_texts(names, preprocess=prep_sec)
    sector_emb_by_country[c2] = (names, emb)

def compute_activity_pack(df_subset: pd.DataFrame) -> Dict[str, Any]:
    df_reset = df_subset.reset_index(drop=True)
    act_emb = embed_texts(df_reset["descriptor"].astype(str).tolist(), preprocess=prep_act)

    gap_hit_list: List[float] = []
    gaps_all_sorted_list: List[np.ndarray] = []
    n_total = len(df_reset)

    for i, row in df_reset.iterrows():
        country = str(row["psilca_country"]).strip().upper()
        _, sec_emb = sector_emb_by_country[country]

        scores_all = act_emb[i] @ sec_emb.T
        max_score = float(np.max(scores_all))

        gaps_all = (max_score - scores_all).astype(np.float32)
        gaps_all_sorted_list.append(np.sort(gaps_all))

        plaus = row["plausible_rank_list"]
        if not plaus:
            gap_hit_list.append(np.inf)
            continue

        best_plaus_score = None
        for r in plaus:
            sj = row.get(f"score_{r}", None)
            if sj is None or (isinstance(sj, float) and np.isnan(sj)):
                continue
            sj = float(sj)
            if (best_plaus_score is None) or (sj > best_plaus_score):
                best_plaus_score = sj

        if best_plaus_score is None:
            gap_hit_list.append(np.inf)
        else:
            gap_hit_list.append(max_score - float(best_plaus_score))

    return {
        "n_total": int(n_total),
        "gap_hit": np.array(gap_hit_list, dtype=np.float32),
        "gaps_all_sorted_list": gaps_all_sorted_list,
        "has_plausible_in_topk": df_reset["has_plausible_in_topk"].to_numpy(bool),
    }

def sweep_delta_utility(pack: Dict[str, Any], deltas: np.ndarray, lam: float) -> pd.DataFrame:
    n_total = pack["n_total"]
    gap_hit = pack["gap_hit"]
    gaps_all_sorted_list = pack["gaps_all_sorted_list"]
    has_plaus = pack["has_plausible_in_topk"]
    n_labeled = int(has_plaus.sum())

    rows = []
    for d in deltas:
        d = float(d)

        hits = (d >= gap_hit)
        recall_all = float(hits.mean()) if n_total else np.nan
        recall_labeled = float(hits[has_plaus].mean()) if n_labeled else np.nan

        retained_counts = np.empty(n_total, dtype=np.int32)
        for i, gaps_sorted in enumerate(gaps_all_sorted_list):
            retained_counts[i] = int(np.searchsorted(gaps_sorted, d, side="right"))

        extra = np.maximum(retained_counts - 1, 0)
        mean_extra = float(extra.mean()) if n_total else np.nan
        median_retained = float(np.median(retained_counts)) if n_total else np.nan
        p90_retained = float(np.percentile(retained_counts, 90)) if n_total else np.nan

        utility_all = float(recall_all - lam * mean_extra) if np.isfinite(recall_all) else np.nan
        utility_labeled = float(recall_labeled - lam * mean_extra) if np.isfinite(recall_labeled) else np.nan

        rows.append({
            "delta": d,
            "recall_all": recall_all,
            "recall_labeled": recall_labeled,
            "mean_extra": mean_extra,
            "median_retained": median_retained,
            "p90_retained": p90_retained,
            "utility_all": utility_all,
            "utility_labeled": utility_labeled,
        })

    return pd.DataFrame(rows)

dev_pack  = compute_activity_pack(dev_high)
test_pack = compute_activity_pack(test_high)

df_delta_dev  = sweep_delta_utility(dev_pack,  DELTA_GRID, LAMBDA_EXTRA)
df_delta_test = sweep_delta_utility(test_pack, DELTA_GRID, LAMBDA_EXTRA)

def choose_delta_by_utility(
    df_dev: pd.DataFrame,
    utility_col: str = "utility_all",
    delta_col: str = "delta"
) -> Tuple[float, str]:
    df = df_dev[[delta_col, utility_col]].dropna().copy()
    if df.empty:
        return float(DELTA_MIN), "fallback: empty utility grid"
    maxu = df[utility_col].max()
    chosen = float(df[df[utility_col] >= maxu - 1e-12].sort_values(delta_col).iloc[0][delta_col])
    rule = f"DEV selection: maximize {utility_col} = Recall − λ*E[(|SΔ|−1)+], tie-breaker=min Δ (λ={LAMBDA_EXTRA})"
    return chosen, rule

chosen_delta, delta_rule = choose_delta_by_utility(df_delta_dev, utility_col="utility_all")
print("Chosen Δ:", chosen_delta)
print("Rule:", delta_rule)


# %% [markdown]
# ## 9) FIGURE 3 (SI): Δ selection (chosen model; high_res) — Regularized utility

# %%
from matplotlib.lines import Line2D

def plot_fig3_delta_utility(df_dev: pd.DataFrame, df_test: pd.DataFrame, chosen_delta: float, lam: float):
    """
    FIG 3 (SI): Δ selection (chosen model; high_res) — matches Fig 1/2 style:
      - Panel headers are centered above each subplot: "(a) ...", "(b) ...", "(c) ..."
      - DEV = solid throughout
      - TEST = hatched throughout (for legend consistency we encode TEST as dashed line + hatched legend patch)
      - Legend below
      - Uses the same spacing constants as your other figs: FIG_TOP, FIG_BOTTOM, LEGEND_Y, PANEL_TITLE_PAD
    """
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), constrained_layout=False)
    fig.subplots_adjust(top=FIG_TOP, bottom=FIG_BOTTOM, wspace=0.35)

    # Line styles (DEV solid; TEST visually distinct)
    dev_style  = dict(linestyle="-",  color=SPLIT_COLORS["DEV"],  linewidth=1.8)
    test_style = dict(linestyle="--", color=SPLIT_COLORS["TEST"], linewidth=1.8)

    xlab = r"$\Delta$ (cosine similarity tolerance)"

    # ---------- (a) Recall vs Δ ----------
    ax = axes[0]
    ax.plot(df_dev["delta"],  df_dev["recall_all"],  **dev_style)
    ax.plot(df_test["delta"], df_test["recall_all"], **test_style)
    ax.axvline(chosen_delta, linestyle=":", linewidth=1.2, color="black")
    panel_header(ax, "a", r"Recall vs $\Delta$")
    ax.set_xlabel(xlab)
    ax.set_ylabel("Recall")
    ax.set_ylim(max(0, min(df_dev["recall_all"].min(), df_test["recall_all"].min()) - 0.02), 1.0)
    nice_axes(ax)

    # ---------- (b) Expected extra retained vs Δ ----------
    ax = axes[1]
    ax.plot(df_dev["delta"],  df_dev["mean_extra"],  **dev_style)
    ax.plot(df_test["delta"], df_test["mean_extra"], **test_style)
    ax.axvline(chosen_delta, linestyle=":", linewidth=1.2, color="black")
    panel_header(ax, "b", r"Penalty vs $\Delta$")
    ax.set_xlabel(xlab)
    ax.set_ylabel(r"$\mathrm{E}[(|S_{\Delta}|-1)_{+}]$")
    ax.set_ylim(0, max(df_dev["mean_extra"].max(), df_test["mean_extra"].max()) * 1.05)
    nice_axes(ax)

    # ---------- (c) Utility vs Δ ----------
    ax = axes[2]
    ax.plot(df_dev["delta"],  df_dev["utility_all"],  **dev_style)
    ax.plot(df_test["delta"], df_test["utility_all"], **test_style)
    ax.axvline(chosen_delta, linestyle=":", linewidth=1.2, color="black")

    # mark Δ* on DEV curve (NO ARROW, NO TEXT)
    row = df_dev.loc[(df_dev["delta"] - chosen_delta).abs().idxmin()]
    ax.scatter([chosen_delta], [row["utility_all"]],
               s=28, zorder=3,
               color=SPLIT_COLORS["DEV"], edgecolor="black", linewidth=0.5)

    panel_header(ax, "c", "Regularized utility")
    ax.set_xlabel(xlab)

    # shorten y-label (long formula goes in caption / SI text)
    ax.set_ylabel("Utility")
    nice_axes(ax)

    # ---- Legend below (consistent placement) ----
    # For bars you use TEST hatch; for lines we mimic that in legend with a hatched patch.
    handles = [
        Line2D([0], [0], color=SPLIT_COLORS["DEV"],  linestyle="-",  linewidth=2.0, label="DEV (solid)"),
        Patch(facecolor="white", edgecolor="black", hatch=HATCH_TEST, label="TEST (hatched)"),
    ]
    add_legend_below(
        fig,
        handles=handles,
        labels=[h.get_label() for h in handles],
        ncol=2,
        title=f"{LABELS.get(CHOSEN_MODEL, CHOSEN_MODEL)} (high-resolution)   Δ*={chosen_delta:.3f}   λ={lam:.2f}",
        y=LEGEND_Y
    )

    save_figure(fig, "Fig3_delta_selection_utility_high_res")
    plt.show()


plot_fig3_delta_utility(df_delta_dev, df_delta_test, chosen_delta, LAMBDA_EXTRA)


# %% [markdown]
# ## 10) Compact report (copy/paste into manuscript/SI)

# %%
chosen_rows = overall[overall["model"] == CHOSEN_MODEL].set_index("split")

dev_row  = df_delta_dev[df_delta_dev["delta"] == chosen_delta].iloc[0]
test_row = df_delta_test[df_delta_test["delta"] == chosen_delta].iloc[0]

print("\n--- Compact report ---")
print(f"Evaluation set sizes: DEV n={n_dev_acts}, TEST n={n_test_acts} unique activities.")
print(f"Chosen semantic model (DEV selection): {CHOSEN_MODEL}")
print(f"  DEV:  Top-1={chosen_rows.loc['dev','top1']:.3f}, Success@{TOP_K}={chosen_rows.loc['dev',f'success@{TOP_K}']:.3f}, MRR={chosen_rows.loc['dev','MRR']:.3f}")
print(f"  TEST: Top-1={chosen_rows.loc['test','top1']:.3f}, Success@{TOP_K}={chosen_rows.loc['test',f'success@{TOP_K}']:.3f}, MRR={chosen_rows.loc['test','MRR']:.3f}")

print(f"\nΔ selection (DEV high_res; regularized utility):")
print(f"  λ = {LAMBDA_EXTRA:.3f}")
print(f"  Chosen Δ = {chosen_delta:.3f}")
print(f"  Rule: {delta_rule}")

print(f"\nAt chosen Δ (DEV high_res):")
print(f"  recall_all={dev_row['recall_all']:.3f}  recall_labeled={dev_row['recall_labeled']:.3f}")
print(f"  mean_extra={dev_row['mean_extra']:.3f}  median_retained={dev_row['median_retained']:.2f}  p90_retained={dev_row['p90_retained']:.2f}")
print(f"  utility_all={dev_row['utility_all']:.3f}  utility_labeled={dev_row['utility_labeled']:.3f}")

print(f"\nAt chosen Δ (TEST high_res):")
print(f"  recall_all={test_row['recall_all']:.3f}  recall_labeled={test_row['recall_labeled']:.3f}")
print(f"  mean_extra={test_row['mean_extra']:.3f}  median_retained={test_row['median_retained']:.2f}  p90_retained={test_row['p90_retained']:.2f}")
print(f"  utility_all={test_row['utility_all']:.3f}  utility_labeled={test_row['utility_labeled']:.3f}")

print("\nTip for SI: report a short sensitivity band:")
dfu = df_delta_dev[["delta","utility_all","recall_all","mean_extra"]].copy()
maxu = dfu["utility_all"].max()
near = dfu[dfu["utility_all"] >= maxu - 0.01].sort_values("delta")
if len(near):
    print(f"  Δ within 0.01 of max utility spans [{near.iloc[0]['delta']:.3f}, {near.iloc[-1]['delta']:.3f}] on DEV high_res.")
else:
    print("  (No near-optimal band found with the chosen threshold.)")
